# ML Methods Comparison Radar Plots

This notebook compares CV AUC across ML classifiers for five model families:

- Clinical
- C-radiomics
- H-radiomics
- DL_3D
- fusion_stacking

Each model family is configured separately. After filling the configuration block, run all cells to collect CV AUCs and generate **five separate radar plots**, one per model family.

In [1]:
# ============================================================
# 1. Imports and global paths
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.metrics import roc_auc_score


# Explicitly register Times New Roman from Windows fonts when available.
times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

MODEL_ROOT = Path('/host/d/projects/Habitats/models/Prognosis')
RESULTS_ROOT = Path('/host/d/projects/Habitats/results')
OUT_DIR = RESULTS_ROOT / 'ML_methods_comparison'
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COL = 'Prognosis_label'
METHOD_ORDER = ['RF', 'LR', 'XGBoost', 'SVM']

print('Model root:', MODEL_ROOT)
print('Output dir:', OUT_DIR)

def save_pdf_and_svg(fig, pdf_path, **kwargs):
    """Save PDF, a normal SVG, and a PPT-friendly SVG copy.

    The normal `.svg` is convenient for conference slides. The `_ppt_safe.svg`
    copy preserves the naming convention used by the other result notebooks.
    """
    pdf_path = Path(pdf_path)
    fig.savefig(pdf_path, **kwargs)

    svg_path = pdf_path.with_suffix('.svg')
    fig.savefig(svg_path, format='svg', **kwargs)

    ppt_safe_svg_path = pdf_path.with_name(pdf_path.stem + '_ppt_safe.svg')
    fig.savefig(ppt_safe_svg_path, format='svg', **kwargs)
    return svg_path, ppt_safe_svg_path

# Backward-compatible alias used by older cells.
def save_pdf_and_ppt_safe_svg(fig, pdf_path, **kwargs):
    return save_pdf_and_svg(fig, pdf_path, **kwargs)


Model root: /host/d/projects/Habitats/models/Prognosis
Output dir: /host/d/projects/Habitats/results/ML_methods_comparison


In [2]:
# ============================================================
# 2. User settings: define each model family separately
# ============================================================

# CV probability mode options:
#   'cv'           -> use prob_cv from cv_predictions.xlsx
#   'allotherdata' -> use prob_cv_allotherdata from cv_predictions.xlsx
#   'advanced'     -> use prob_cv_final_advanced from cv_predictions.xlsx
#   'final'        -> if this method is the family-level final-selection method,
#                     read final_selections/cv_final_selection_predictions.xlsx;
#                     otherwise read cv_metrics.xlsx and choose prob_cv or prob_cv_allotherdata
#                     according to the experiment-level selected_method.
#
# Notes:
# - Fill MANUAL_EXPERIMENTS for methods that are not using family-level final selection.
# - XGBoost can stay in METHOD_ORDER even if you use another folder as a substitute;
#   just map 'XGBoost' to whichever folder you want in METHOD_DIR_MAP.
# - For fusion_stacking, final_selection_dir usually includes the final method subfolder,
#   e.g. /fusion/stacking/final_selections/RF.

MODEL_CONFIGS = {
    'Clinical': {
        'model_root': MODEL_ROOT / 'clinical',
        'final_selection_dir': MODEL_ROOT / 'clinical' / 'final_selections',
        'final_selection_method': 'RF',
        'method_dir_map': {
            'RF': 'RandomForest',
            'LR': 'LR',
            'XGBoost': 'SVM',  # user-selected substitute for clinical XGBoost axis
            'SVM': 'SVM',
        },
        'manual_experiments': {
            'RF': '',
            'LR': 'random40_rfecv_none',
            'XGBoost': 'random0_rfecv_none',
            'SVM': 'random10_rfecv_none',
        },
        'cv_prob_mode': {
            'RF': 'final',
            'LR': 'advanced',
            'XGBoost': 'advanced',
            'SVM': 'advanced',
        },
    },

    'C-radiomics': {
        'model_root': MODEL_ROOT / 'whole_image',
        'final_selection_dir': MODEL_ROOT / 'whole_image' / 'final_selections',
        'final_selection_method': 'SVM',
        'method_dir_map': {
            'RF': 'RandomForest',
            'LR': 'LR',
            'XGBoost': 'XGBoost',
            'SVM': 'SVM',
        },
        'manual_experiments': {
            'RF': 'random30_rfe_top7',
            'LR': 'random0_lasso_top27',
            'XGBoost': 'random30_rfe_top17',
            'SVM': '',
        },
        'cv_prob_mode': {
            'RF': 'advanced',
            'LR': 'advanced',
            'XGBoost': 'advanced',
            'SVM': 'final',
        },
    },

    'H-radiomics': {
        'model_root': MODEL_ROOT / 'habitats_avg',
        'final_selection_dir': MODEL_ROOT / 'habitats_avg' / 'final_selections',
        'final_selection_method': 'SVM',
        'method_dir_map': {
            'RF': 'RandomForest',
            'LR': 'LR',
            'XGBoost': 'XGBoost',
            'SVM': 'SVM',
        },
        'manual_experiments': {
            'RF': 'random20_rfe_top12',
            'LR': 'random30_lasso_top15',
            'XGBoost': 'random10_rfe_top12',
            'SVM': '',
        },
        'cv_prob_mode': {
            'RF': 'advanced',
            'LR': 'advanced',
            'XGBoost': 'advanced',
            'SVM': 'final',
        },
    },

    'DL_3D': {
        'model_root': MODEL_ROOT / 'dl_3d_ml_all',
        'final_selection_dir': MODEL_ROOT / 'dl_3d_ml_all' / 'final_selections',
        'final_selection_method': 'LR',
        'method_dir_map': {
            'RF': 'RandomForest',
            'LR': 'LR',
            'XGBoost': 'XGBoost',
            'SVM': 'SVM',
        },
        'manual_experiments': {
            'RF': 'random20_lasso_top6',
            'LR': '',
            'XGBoost': 'random20_lasso_top8',
            'SVM': 'random40_rfe_top6',
        },
        'cv_prob_mode': {
            'RF': 'final',
            'LR': 'final',
            'XGBoost': 'final',
            'SVM': 'final',
        },
    },

    'fusion_stacking': {
        'model_root': MODEL_ROOT / 'fusion' / 'stacking',
        'final_selection_dir': MODEL_ROOT / 'fusion' / 'stacking' / 'final_selections' / 'RF',
        'final_selection_method': 'RF',  # e.g. 'RF'
        'method_dir_map': {
            'RF': 'RF',
            'LR': 'LR',
            'XGBoost': 'XGBOOST',
            'SVM': 'SVM',
        },
        'manual_experiments': {
            'RF': '',       # e.g. 'random20'
            'LR': 'random0',       # e.g. 'random30'
            'XGBoost': 'random10',  # e.g. 'random10'
            'SVM': 'random20',      # e.g. 'random0'
        },
        'cv_prob_mode': {
            'RF': 'final',
            'LR': 'final',
            'XGBoost': 'final',
            'SVM': 'final',
        },
    },
}

# One color per model family. These follow the previous palette.
MODEL_COLORS = {
    'Clinical': '#4C78A8',
    'C-radiomics': '#72B7B2',
    'H-radiomics': '#9D7CC1',
    'DL_3D': '#C44E52',
    'fusion_stacking': '#E39C37',
}

# Each model gets its own radial axis range.
# Step=0.05 keeps the ticks paper-friendly; min_span prevents over-zooming.
RADIAL_STEP = 0.01
MIN_RADIAL_SPAN = 0.04
RADIAL_MARGIN = 0.005
MAX_RADIAL_TICKS = 6

FIGSIZE = (5.0, 4.8)
SAVE_PREFIX = 'ML_methods_CV_AUC_radar'


In [3]:
# ============================================================
# 3. Helper functions
# ============================================================

def require_nonempty(value, name):
    if value is None or str(value).strip() == '':
        raise ValueError(f'Please fill {name} in the configuration block.')
    return str(value).strip()


def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def infer_selected_experiment_for_method(config, method):
    manifest_path = Path(config['final_selection_dir']) / 'cv_final_selection_manifest.json'
    if not manifest_path.exists():
        return None
    manifest = read_json(manifest_path)
    for item in manifest.get('selected_cv_experiments', []):
        if item.get('classifier') == method:
            return item.get('experiment')
    return None


def get_experiment_for_method(model_name, config, method):
    experiment = str(config['manual_experiments'].get(method, '')).strip()
    if experiment:
        return experiment

    final_method = str(config.get('final_selection_method', '')).strip()
    if method == final_method:
        experiment = infer_selected_experiment_for_method(config, method)
        if experiment:
            return experiment

    raise ValueError(
        f'[{model_name}] No experiment defined for {method}. '
        f'Fill MODEL_CONFIGS["{model_name}"]["manual_experiments"]["{method}"].'
    )


def method_folder(config, method):
    folder = str(config['method_dir_map'].get(method, '')).strip()
    require_nonempty(folder, f'method_dir_map[{method}]')
    return folder


def cv_prediction_file_for_experiment(config, method, experiment):
    path = Path(config['model_root']) / method_folder(config, method) / experiment / 'cv_predictions.xlsx'
    if not path.exists():
        raise FileNotFoundError(f'CV prediction file not found: {path}')
    return path


def cv_metrics_file_for_experiment(config, method, experiment):
    path = Path(config['model_root']) / method_folder(config, method) / experiment / 'cv_metrics.xlsx'
    if not path.exists():
        raise FileNotFoundError(f'CV metrics file not found: {path}')
    return path


def final_selection_cv_prediction_file(config):
    path = Path(config['final_selection_dir']) / 'cv_final_selection_predictions.xlsx'
    if not path.exists():
        raise FileNotFoundError(f'Final-selection CV prediction file not found: {path}')
    return path


def final_selection_probability_column(df):
    for col in ['prob_final_selection', 'prob_mix', 'prob_mean']:
        if col in df.columns:
            return col
    raise KeyError(f'No final-selection CV probability column found. Available columns: {df.columns.tolist()}')


def cv_probability_column_for_mode(config, method, experiment, mode, df):
    mode = str(mode).lower().strip()
    if mode == 'cv':
        col = 'prob_cv'
    elif mode == 'allotherdata':
        col = 'prob_cv_allotherdata'
    elif mode == 'advanced':
        col = 'prob_cv_final_advanced'
    elif mode == 'final':
        metrics_path = cv_metrics_file_for_experiment(config, method, experiment)
        metrics_df = pd.read_excel(metrics_path)

        # Two ML code generations use slightly different cv_metrics schemas:
        #   radiomics/clinical/DL-ML: name == 'final'
        #   fusion stacking:         mode == 'cv_final'
        if 'name' in metrics_df.columns:
            final_row = metrics_df[metrics_df['name'].astype(str).str.lower() == 'final']
        elif 'mode' in metrics_df.columns:
            final_row = metrics_df[metrics_df['mode'].astype(str).str.lower().isin(['cv_final', 'final'])]
        else:
            raise KeyError(f'Neither name nor mode column found in {metrics_path}. Columns: {metrics_df.columns.tolist()}')

        if final_row.empty:
            raise RuntimeError(f'No final row found in {metrics_path}')

        # Prefer the explicit probability_column if present. This handles fusion stacking,
        # where cv_final may be saved as prob_cv_final.
        explicit_col = str(final_row.iloc[-1].get('probability_column', '')).strip()
        if explicit_col and explicit_col in df.columns:
            col = explicit_col
        else:
            selected_method = str(final_row.iloc[-1].get('selected_method', '')).lower()
            if selected_method in {'allotherdata', 'all_other_data'}:
                col = 'prob_cv_allotherdata'
            elif selected_method in {'together', 'cv', 'traditional'}:
                col = 'prob_cv'
            else:
                raise RuntimeError(f'Unknown selected_method={selected_method!r} in {metrics_path}')
    else:
        raise ValueError(f'Unknown CV mode: {mode}. Allowed: cv, allotherdata, final, advanced')

    if col not in df.columns:
        raise KeyError(f'Column {col} not found. Available columns: {df.columns.tolist()}')
    return col


def read_cv_prediction(model_name, config, method):
    mode = require_nonempty(config['cv_prob_mode'].get(method, ''), f'{model_name}.cv_prob_mode[{method}]').lower()
    final_method = str(config.get('final_selection_method', '')).strip()

    if method == final_method and mode == 'final':
        path = final_selection_cv_prediction_file(config)
        df = pd.read_excel(path)
        prob_col = final_selection_probability_column(df)
        experiment = 'final_selection'
        source = f'{Path(config["final_selection_dir"]).name}/{path.name}'
    else:
        experiment = get_experiment_for_method(model_name, config, method)
        path = cv_prediction_file_for_experiment(config, method, experiment)
        df = pd.read_excel(path)
        prob_col = cv_probability_column_for_mode(config, method, experiment, mode, df)
        source = f'{method_folder(config, method)}/{experiment}/{path.name}'

    if LABEL_COL not in df.columns:
        raise KeyError(f'{LABEL_COL} not found in {path}')

    y = df[LABEL_COL].astype(int).to_numpy()
    p = df[prob_col].astype(float).to_numpy()
    auc = roc_auc_score(y, p)

    return {
        'Model': model_name,
        'Method': method,
        'Experiment': experiment,
        'CV_mode': mode,
        'Probability_column': prob_col,
        'Prediction_file': str(path),
        'AUC': auc,
        'N': len(df),
        'Positive_n': int(y.sum()),
        'Positive_fraction': float(y.mean()),
        'Source': source,
    }


def collect_all_auc_rows():
    rows = []
    for model_name, config in MODEL_CONFIGS.items():
        for method in METHOD_ORDER:
            rows.append(read_cv_prediction(model_name, config, method))
    return pd.DataFrame(rows)


def radial_limits(values, step=RADIAL_STEP, margin=RADIAL_MARGIN, min_span=MIN_RADIAL_SPAN):
    """Return a tight per-model radial range to emphasize within-model AUC differences."""
    values = np.asarray(values, dtype=float)
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))

    lower = np.floor((vmin - margin) / step) * step
    upper = np.ceil((vmax + margin) / step) * step

    # Keep a small minimum span so tiny differences are visible but not absurdly magnified.
    if upper - lower < min_span:
        center = (vmin + vmax) / 2
        lower = np.floor((center - min_span / 2) / step) * step
        upper = np.ceil((center + min_span / 2) / step) * step

    lower = max(0.0, lower)
    upper = min(1.0, upper)
    return float(lower), float(upper)


def radial_ticks(lower, upper, max_ticks=MAX_RADIAL_TICKS):
    """Create readable tick labels for a tight AUC axis."""
    span = upper - lower
    if span <= 0.06:
        tick_step = 0.01
    elif span <= 0.12:
        tick_step = 0.02
    else:
        tick_step = 0.05

    ticks = np.arange(lower, upper + tick_step / 2, tick_step)
    if len(ticks) > max_ticks:
        ticks = np.linspace(lower, upper, max_ticks)
    return ticks


def close_loop(values):
    values = list(values)
    return values + values[:1]


def safe_filename(text):
    return str(text).replace(' ', '_').replace('/', '_')


In [4]:
# ============================================================
# 4. Collect CV AUC data
# ============================================================

auc_long_df = collect_all_auc_rows()
auc_wide_df = (
    auc_long_df
    .pivot(index='Model', columns='Method', values='AUC')
    .reindex(list(MODEL_CONFIGS.keys()))
    .reindex(columns=METHOD_ORDER)
)

long_path = OUT_DIR / f'{SAVE_PREFIX}_auc_long.xlsx'
wide_path = OUT_DIR / f'{SAVE_PREFIX}_auc_wide.xlsx'
auc_long_df.to_excel(long_path, index=False)
auc_wide_df.reset_index().to_excel(wide_path, index=False)

print('Saved:', long_path)
print('Saved:', wide_path)
display(auc_long_df)
display(auc_wide_df)


Saved: /host/d/projects/Habitats/results/ML_methods_comparison/ML_methods_CV_AUC_radar_auc_long.xlsx
Saved: /host/d/projects/Habitats/results/ML_methods_comparison/ML_methods_CV_AUC_radar_auc_wide.xlsx


,Model,Method,Experiment,CV_mode,Probability_column,Prediction_file,AUC,N,Positive_n,Positive_fraction,Source
0,Clinical,RF,final_selection,final,prob_final_selection,/host/d/projects/Habitats/models/Prognosis/cli...,0.692997,188,49,0.260638,final_selections/cv_final_selection_prediction...
1,Clinical,LR,random40_rfecv_none,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/cli...,0.644986,188,49,0.260638,LR/random40_rfecv_none/cv_predictions.xlsx
2,Clinical,XGBoost,random0_rfecv_none,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/cli...,0.588460,188,49,0.260638,SVM/random0_rfecv_none/cv_predictions.xlsx
3,Clinical,SVM,random10_rfecv_none,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/cli...,0.634268,188,49,0.260638,SVM/random10_rfecv_none/cv_predictions.xlsx
4,C-radiomics,RF,random30_rfe_top7,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/who...,0.738218,188,49,0.260638,RandomForest/random30_rfe_top7/cv_predictions....
5,C-radiomics,LR,random0_lasso_top27,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/who...,0.754808,188,49,0.260638,LR/random0_lasso_top27/cv_predictions.xlsx
6,C-radiomics,XGBoost,random30_rfe_top17,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/who...,0.772721,188,49,0.260638,XGBoost/random30_rfe_top17/cv_predictions.xlsx
7,C-radiomics,SVM,final_selection,final,prob_final_selection,/host/d/projects/Habitats/models/Prognosis/who...,0.783732,188,49,0.260638,final_selections/cv_final_selection_prediction...
8,H-radiomics,RF,random20_rfe_top12,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/hab...,0.704889,188,49,0.260638,RandomForest/random20_rfe_top12/cv_predictions...
9,H-radiomics,LR,random30_lasso_top15,advanced,prob_cv_final_advanced,/host/d/projects/Habitats/models/Prognosis/hab...,0.742035,188,49,0.260638,LR/random30_lasso_top15/cv_predictions.xlsx


Method,RF,LR,XGBoost,SVM
Model,,,,
Clinical,0.692997,0.644986,0.588460,0.634268
C-radiomics,0.738218,0.754808,0.772721,0.783732
H-radiomics,0.704889,0.742035,0.773675,0.817061
DL_3D,0.808251,0.844223,0.803994,0.845838
fusion_stacking,0.905741,0.902070,0.901483,0.882616


In [5]:
# ============================================================
# 5. Save one radar plot per model family, plus a combined layout
# ============================================================

labels = METHOD_ORDER
n = len(labels)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles_closed = angles + angles[:1]


def draw_radar(ax, model_name, values, color, show_title=True, title_y=1.12):
    """Draw one model-family radar chart on an existing polar axis."""
    y_min, y_max = radial_limits(values)
    ticks = radial_ticks(y_min, y_max)

    ax.plot(
        angles_closed,
        close_loop(values),
        color=color,
        linewidth=2.2,
    )
    ax.fill(
        angles_closed,
        close_loop(values),
        color=color,
        alpha=0.25,
    )
    ax.scatter(angles, values, s=34, color=color, zorder=3)

    ax.set_xticks(angles)
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylim(y_min, y_max)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f'{t:.2f}' for t in ticks], fontsize=11)
    ax.set_rlabel_position(88)
    ax.grid(True, linestyle=':', linewidth=0.8, alpha=0.85)
    ax.spines['polar'].set_color('#888888')
    ax.spines['polar'].set_linewidth(0.8)

    if show_title:
        ax.set_title(model_name, y=title_y, fontsize=18, fontweight='bold')

    return y_min, y_max


saved_plots = []

# ------------------------------------------------------------
# 5A. Individual figures: one model family per file
# ------------------------------------------------------------
for model_name in MODEL_CONFIGS.keys():
    values = auc_wide_df.loc[model_name, METHOD_ORDER].astype(float).to_list()
    color = MODEL_COLORS.get(model_name, '#4C78A8')

    fig = plt.figure(figsize=FIGSIZE)
    ax = plt.subplot(111, polar=True)
    y_min, y_max = draw_radar(ax, model_name, values, color, show_title=True, title_y=1.10)

    plt.tight_layout()

    base = f'{SAVE_PREFIX}_{safe_filename(model_name)}'
    pdf_path = OUT_DIR / f'{base}.pdf'
    png_path = OUT_DIR / f'{base}.png'
    svg_path, ppt_svg_path = save_pdf_and_ppt_safe_svg(fig, pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.show()

    saved_plots.append({
        'Model': model_name,
        'PDF': str(pdf_path),
        'SVG': str(svg_path),
        'PPT_safe_SVG': str(ppt_svg_path),
        'PNG': str(png_path),
        'y_min': y_min,
        'y_max': y_max,
    })


# ------------------------------------------------------------
# 5B. Combined figure: 3 panels on the first row, 2 centered below
# ------------------------------------------------------------
combined_fig = plt.figure(figsize=(13.2, 8.4))
gs = combined_fig.add_gridspec(
    2,
    6,
    left=0.04,
    right=0.98,
    top=0.94,
    bottom=0.06,
    wspace=0.62,
    hspace=0.58,
)

# Top row: 3 equally spaced panels. Bottom row: 2 centered panels.
combined_positions = [(0, slice(0, 2)), (0, slice(2, 4)), (0, slice(4, 6)), (1, slice(1, 3)), (1, slice(3, 5))]

for (model_name, position) in zip(MODEL_CONFIGS.keys(), combined_positions):
    row, col_slice = position
    ax = combined_fig.add_subplot(gs[row, col_slice], polar=True)
    values = auc_wide_df.loc[model_name, METHOD_ORDER].astype(float).to_list()
    color = MODEL_COLORS.get(model_name, '#4C78A8')
    draw_radar(ax, model_name, values, color, show_title=True, title_y=1.13)

combined_pdf_path = OUT_DIR / f'{SAVE_PREFIX}_combined.pdf'
combined_png_path = OUT_DIR / f'{SAVE_PREFIX}_combined.png'
combined_svg_path, combined_ppt_svg_path = save_pdf_and_ppt_safe_svg(combined_fig, combined_pdf_path, bbox_inches='tight')
combined_fig.savefig(combined_png_path, dpi=300, bbox_inches='tight')
plt.show()

saved_plots.append({
    'Model': 'combined',
    'PDF': str(combined_pdf_path),
    'SVG': str(combined_svg_path),
    'PPT_safe_SVG': str(combined_ppt_svg_path),
    'PNG': str(combined_png_path),
    'y_min': '',
    'y_max': '',
})

saved_plot_df = pd.DataFrame(saved_plots)
saved_plot_df.to_excel(OUT_DIR / f'{SAVE_PREFIX}_saved_plots.xlsx', index=False)
print('Saved radar plots to:', OUT_DIR)
display(saved_plot_df)


/tmp/ipykernel_61064/706007241.py:66: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


/tmp/ipykernel_61064/706007241.py:66: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


/tmp/ipykernel_61064/706007241.py:66: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


/tmp/ipykernel_61064/706007241.py:66: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


/tmp/ipykernel_61064/706007241.py:66: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved radar plots to: /host/d/projects/Habitats/results/ML_methods_comparison


/tmp/ipykernel_61064/706007241.py:108: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,Model,PDF,SVG,PPT_safe_SVG,PNG,y_min,y_max
0,Clinical,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,0.58,0.7
1,C-radiomics,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,0.73,0.79
2,H-radiomics,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,0.69,0.83
3,DL_3D,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,0.79,0.86
4,fusion_stacking,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,0.87,0.92
5,combined,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,/host/d/projects/Habitats/results/ML_methods_c...,,
